In [1]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
import anndata
import time
import matplotlib.pyplot as plt
import json

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

pd.set_option('display.max_columns', 500)

In [2]:
version = '20260711'
download_base = Path('../../data/allen-brain-cell-atlas-staging')
abc_cache = AbcProjectCache.from_cache_dir(
    download_base,
    s3_bucket='allen-brain-cell-atlas-staging',
    auth_required=True,
)

abc_cache.load_manifest(f'releases/{version}/manifest.json')

Read in the two DataFrames from the aging dataset we'll need to create an equivalent cluster annotation terms and term set like the WMB and WHB taxonomies.

In [3]:
taxonomy_dir = 'SEA-AD-CaH-taxonomy'

In [4]:
abc_cache.list_metadata_files(taxonomy_dir)

['cell_2d_embedding_coordinates',
 'cell_to_cluster_membership',
 'cluster',
 'cluster_annotation_term',
 'cluster_annotation_term_set',
 'cluster_to_cluster_annotation_membership']

In [5]:
term = abc_cache.get_metadata_dataframe(
    taxonomy_dir,
    'cluster_annotation_term'
)
term

cluster_annotation_term.csv: 100%|██████████| 12.3k/12.3k [00:00<00:00, 93.4kMB/s]


,label,name,cluster_annotation_term_set_label,cluster_annotation_term_set_name,color_hex_triplet,term_order,term_set_order,parent_term_label,parent_term_name,parent_term_set_label,CCN20230508_label,CCN20250428_label
0,CS20260701_NEIG_04,Astro-Epen,CCN20260701_LEVEL_0,Neighborhood,#9A7345,4,0,NaN,NaN,NaN,NaN,NaN
1,CS20260701_NEIG_03,CGE,CCN20260701_LEVEL_0,Neighborhood,#C355B0,3,0,NaN,NaN,NaN,NaN,NaN
2,CS20260701_NEIG_05,Immune,CCN20260701_LEVEL_0,Neighborhood,#6BBC6B,5,0,NaN,NaN,NaN,NaN,NaN
3,CS20260701_NEIG_01,LGE,CCN20260701_LEVEL_0,Neighborhood,#5B47C1,1,0,NaN,NaN,NaN,NaN,NaN
4,CS20260701_NEIG_08,LSX,CCN20260701_LEVEL_0,Neighborhood,#B1B1B1,8,0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
85,CS20260701_SUPR_30,Vip_23,CCN20260701_LEVEL_2,Supertype,#5A3D67,30,2,CS20260701_SCLA_08,Vip,CCN20260701_LEVEL_1,CS20230508_SUPT_0034,NaN
86,CS20260701_SUPR_31,Vip_4,CCN20260701_LEVEL_2,Supertype,#D0AEDC,31,2,CS20260701_SCLA_08,Vip,CCN20260701_LEVEL_1,CS20230508_SUPT_0022,NaN
87,CS20260701_SUPR_32,Vip_5,CCN20260701_LEVEL_2,Supertype,#C7A5D3,32,2,CS20260701_SCLA_08,Vip,CCN20260701_LEVEL_1,CS20230508_SUPT_0023,NaN
88,CS20260701_SUPR_33,Vip_6,CCN20260701_LEVEL_2,Supertype,#BE9DCA,33,2,CS20260701_SCLA_08,Vip,CCN20260701_LEVEL_1,CS20230508_SUPT_0024,NaN


In [6]:
term_sets = abc_cache.get_metadata_dataframe(
    directory=taxonomy_dir,
    file_name='cluster_annotation_term_set'
).set_index('label')
term_sets

cluster_annotation_term_set.csv:   0%|          | 0.00/222 [00:00<?, ?MB/s]

cluster_annotation_term_set.csv: 100%|██████████| 222/222 [00:00<00:00, 2.10kMB/s]


,name,description,order,parent_term_set_label
label,,,,
CCN20260701_LEVEL_0,Neighborhood,Neighborhood,0,NaN
CCN20260701_LEVEL_1,Subclass,Subclass,1,CCN20260701_LEVEL_0
CCN20260701_LEVEL_2,Supertype,Supertype,2,CCN20260701_LEVEL_1


In [7]:
filtered = term[pd.notna(term['parent_term_label'])]
first_child = filtered.groupby('parent_term_label')[['label','name','term_order','cluster_annotation_term_set_name']].first()
first_child

,label,name,term_order,cluster_annotation_term_set_name
parent_term_label,,,,
CS20260701_NEIG_01,CS20260701_SCLA_01,STR D1 MSN,1,Subclass
CS20260701_NEIG_02,CS20260701_SCLA_04,CN ST18 GABA,4,Subclass
CS20260701_NEIG_03,CS20260701_SCLA_07,CN LAMP5-CXCL14 GABA,7,Subclass
CS20260701_NEIG_04,CS20260701_SCLA_09,Astrocyte,9,Subclass
CS20260701_NEIG_05,CS20260701_SCLA_11,Microglia-PVM,11,Subclass
CS20260701_NEIG_06,CS20260701_SCLA_12,OPC,12,Subclass
CS20260701_NEIG_07,CS20260701_SCLA_14,Endothelial,14,Subclass
CS20260701_NEIG_08,CS20260701_SCLA_16,LSX,16,Subclass
CS20260701_SCLA_01,CS20260701_SUPR_01,STRd D1 Matrix MSN,1,Supertype


In [8]:
term.set_index('label',inplace=True)
term.loc[first_child.index,'first_child_label'] = first_child['label']
term.loc[first_child.index,'first_child_term_set_name'] = first_child['cluster_annotation_term_set_name']
term.reset_index(inplace=True)

In [9]:
term[pd.notna(term['first_child_label'])]

,label,name,cluster_annotation_term_set_label,cluster_annotation_term_set_name,color_hex_triplet,term_order,term_set_order,parent_term_label,parent_term_name,parent_term_set_label,CCN20230508_label,CCN20250428_label,first_child_label,first_child_term_set_name
0,CS20260701_NEIG_04,Astro-Epen,CCN20260701_LEVEL_0,Neighborhood,#9A7345,4,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_09,Subclass
1,CS20260701_NEIG_03,CGE,CCN20260701_LEVEL_0,Neighborhood,#C355B0,3,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_07,Subclass
2,CS20260701_NEIG_05,Immune,CCN20260701_LEVEL_0,Neighborhood,#6BBC6B,5,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_11,Subclass
3,CS20260701_NEIG_01,LGE,CCN20260701_LEVEL_0,Neighborhood,#5B47C1,1,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_01,Subclass
4,CS20260701_NEIG_08,LSX,CCN20260701_LEVEL_0,Neighborhood,#B1B1B1,8,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_16,Subclass
5,CS20260701_NEIG_02,MGE,CCN20260701_LEVEL_0,Neighborhood,#BBB24D,2,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_04,Subclass
6,CS20260701_NEIG_06,OPC-Oligo,CCN20260701_LEVEL_0,Neighborhood,#459A8C,6,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_12,Subclass
7,CS20260701_NEIG_07,Vascular,CCN20260701_LEVEL_0,Neighborhood,#BF6958,7,0,NaN,NaN,NaN,NaN,NaN,CS20260701_SCLA_14,Subclass
8,CS20260701_SCLA_09,Astrocyte,CCN20260701_LEVEL_1,Subclass,#665C47,9,1,CS20260701_NEIG_04,Astro-Epen,CCN20260701_LEVEL_0,NaN,NaN,CS20260701_SUPR_35,Supertype
9,CS20260701_SCLA_07,CN LAMP5-CXCL14 GABA,CCN20260701_LEVEL_1,Subclass,#e5b672,7,1,CS20260701_NEIG_03,CGE,CCN20260701_LEVEL_0,NaN,NaN,CS20260701_SUPR_17,Supertype


In [10]:
membership = abc_cache.get_metadata_dataframe(directory=taxonomy_dir, file_name='cluster_to_cluster_annotation_membership')
pivot = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])['cluster_annotation_term_name'].first().unstack()
pivot = pivot[term_sets['name']] # order columns
pivot.fillna('Other', inplace=True)
pivot.sort_values(['Neighborhood', 'Subclass', 'Supertype'], inplace=True)
cols = pivot.columns.to_list()
pivot.columns = cols
pivot

cluster_to_cluster_annotation_membership.csv: 100%|██████████| 12.4k/12.4k [00:00<00:00, 112kMB/s] 


,Neighborhood,Subclass,Supertype
cluster_alias,,,
35,Astro-Epen,Astrocyte,Astro_1
36,Astro-Epen,Astrocyte,Astro_2
37,Astro-Epen,Astrocyte,Astro_3
38,Astro-Epen,Astrocyte,Astro_4
39,Astro-Epen,Astrocyte,Astro_5
...,...,...,...
61,Vascular,Endothelial,Endo_3
62,Vascular,VLMC,Pericyte_1
63,Vascular,VLMC,Pericyte_2-SEAAD


In [11]:
lookup = {}
for tag in term_sets['name']:
    #print(tag)
    pred = (term['cluster_annotation_term_set_name'] == tag)
    filtered = term[pred].copy()
    filtered.set_index('name', inplace=True)
    lookup[tag] = filtered

Helper functions to lookup an term attribut and format a cell in the html table

In [12]:
def get_value(c, n, v) :
    return lookup[c].loc[n][v]

def format_cell (df,c,add_id=False,add_plus=False,add_minus=False) :

    divs = pd.DataFrame(index=df.index)
    
    pattern = '<div class="circle" style="background-color:%s"></div>'
    divs['circle'] = [pattern % get_value(c,x,'color_hex_triplet') for x in df[c]]
    
    pattern = '<div class="celltext">%s</div>'
    divs['name'] = [pattern % x for x in df[c]]
   
    divs['id'] = ''
    if add_id :
        pattern = '<div id="%s"></div>'
        divs['id'] = [pattern % get_value(c,x,'label') for x in df[c]]
        
    divs['plus'] = ''
    if add_plus :
        pattern = '<div class="celltext"><a href="%s.html#%s">[+]</a></div>'
        divs['plus'] = [pattern % (get_value(c,x,'first_child_term_set_name'),
                                   get_value(c,x,'first_child_label')) for x in df[c]]
        
    divs['minus'] = ''
    if add_minus :
        pattern = '<div class="celltext"><a href="%s.html#%s">[-]</a></div>'
        divs['minus'] = [pattern % (get_value(c,x,'cluster_annotation_term_set_name'),
                                    get_value(c,x,'label')) for x in df[c]]
    
    cols = ['id','circle','name','plus','minus']
    output = divs[cols].apply(lambda row: ''.join(row.values.astype(str)), axis=1)
    return output


Helper function to create html document

In [13]:
def create_html(df, ts, file, title):
    
    # apply formatter to each term set
    df_formatted = df.copy()
    
    for tag in term_sets['name'] :
        if tag in df_formatted.columns :

            add_id = False
            if tag == ts :
                add_id = True
                
            add_plus = False
            if tag == ts and tag not in ['Supertype']:
                add_plus = True
                
            add_minus = False
            if tag != ts and tag not in ['']:
                add_minus = True
                
            df_formatted[tag] = format_cell(df,tag,add_id,add_plus,add_minus)
            
            
    output = df_formatted.to_html(index=False, na_rep='',
                        render_links=True,escape=False,
                        classes="mystyle")

    html_string = '''
    <html>
    <head><title>%s</title></head>
    <link rel="stylesheet" type="text/css" href="../../simple_style.css"/>
    <body>
    {table}
    </body>
    </html>.
    ''' % title

    # OUTPUT AN HTML FILE
    with open(file, 'w') as f:
        f.write(html_string.format(table=output))

In [14]:
# Write the data to the _static directory of the abc_atlas_access so that links work properly in the jupyter-book/sphinx page.
output_directory = os.path.join('../../_static', taxonomy_dir, version)
os.makedirs(output_directory, exist_ok=True)

In [17]:
df_supertype = pivot[['Neighborhood']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'neighborhood.html')
title = 'SEA-AD CaH taxonomy: Neighborhood'
create_html(df_supertype, 'Neighborhood', file, title)
print(len(df_supertype))

8


In [18]:
df_supertype = pivot[['Neighborhood', 'Subclass']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'subclass.html')
title = 'SEA-AD CaH taxonomy: Subclass'
create_html(df_supertype, 'Subclass', file, title)
print(len(df_supertype))

16


In [19]:
df_supertype = pivot[['Neighborhood', 'Subclass', 'Supertype']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'supertype.html')
title = 'SEA-AD Cah taxonomy: Supertype'
create_html(df_supertype, 'Supertype', file, title)
print(len(df_supertype))

66
